# Self-Supervised Contrastive CNN + Prototype Similarity

Этот ноутбук строит **нейросетевую карту перспективности без обычного supervised-обучения**.

Идея метода:

1. Вся территория переводится в регулярную сетку `500 x 500 м`.
2. Для каждой ячейки считаются базовые геологические каналы.
3. Из карты режутся `patch`-окрестности вокруг ячеек.
4. На этих `patch` обучается **self-supervised contrastive CNN**.
5. После обучения для `patch` вокруг известных проявлений считаются `embedding`.
6. Средний `embedding` положительных ячеек берётся как **positive prototype**.
7. Для всех ячеек считается **cosine similarity** к этому prototype.
8. Похожесть интерпретируется как `prospectivity`, а затем строится карта `prognoz = 1 - prospectivity`.

Это не методичка и не псевдометки.  
Сеть учится **на всей карте**, а редкие точки используются только как **малый опорный набор**.

In [ ]:

# Если torch не установлен, раскомментируйте:
# !pip install torch

In [ ]:

from pathlib import Path
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from shapely.ops import unary_union

from sklearn.preprocessing import RobustScaler
from scipy.signal import convolve2d
from scipy.ndimage import maximum_filter, label

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:

# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз")
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "ss_contrastive_proto_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CELL_SIZE = 500

LAYER_MAP = {
    "mask": "svita_new",
    "facies": "fasii",
    "paleo": "gr_dol_vp_poly",
    "struct": "kory",
    "magm": "dayki_buf",
    "tect1": "glub_raz_nw",
    "tect2": "glub_r_nw",
}

# Если None -> берем все point-слои-кандидаты
POSITIVE_LAYER_NAMES = None
# Пример:
# POSITIVE_LAYER_NAMES = ["result", "layer_01"]

PATCH_SIZE = 11
MAX_TRAIN_PATCHES = 8000
BATCH_SIZE = 128
EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
TEMPERATURE = 0.15
EMBED_DIM = 32
SEED = 42

SMOOTH_PASSES = 2
TOP_ZONE_Q = 0.992
LOCAL_MAX_SIZE = 5
MIN_TOP_COMPONENT_CELLS = 2

SCALES = {
    "facies": 1200.0,
    "paleo": 1200.0,
    "struct": 900.0,
    "magm": 900.0,
    "tect1": 1000.0,
    "tect2": 1000.0,
}

SHOW_POINTS = True
SAVE_FULL_GRID_GPKG = True

In [ ]:

# =========================
# REPRODUCIBILITY
# =========================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", DEVICE)

In [ ]:

# =========================
# HELPERS
# =========================
def repair_geometries(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    gdf = gdf[~gdf.geometry.isna()].copy()
    if len(gdf) == 0:
        return gdf
    try:
        gdf["geometry"] = gdf.geometry.buffer(0)
    except Exception:
        pass
    gdf = gdf[~gdf.geometry.is_empty].copy()
    return gdf


def load_shapefiles(folder: Path):
    layers = {}
    for shp in sorted(folder.glob("*.shp")):
        name = shp.stem
        gdf = gpd.read_file(shp)
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326, allow_override=True)
        gdf = repair_geometries(gdf)
        layers[name] = gdf
    return layers


def choose_target_crs(mask_gdf: gpd.GeoDataFrame):
    crs = mask_gdf.crs
    if crs is None:
        return "EPSG:3857"
    try:
        if crs.is_projected:
            return crs
    except Exception:
        pass
    try:
        utm = mask_gdf.estimate_utm_crs()
        if utm is not None:
            return utm
    except Exception:
        pass
    return "EPSG:3857"


def harmonize_layers(layers: dict, target_crs):
    out = {}
    for name, gdf in layers.items():
        cur = gdf.copy()
        if cur.crs is None:
            cur = cur.set_crs(epsg=4326, allow_override=True)
        if str(cur.crs) != str(target_crs):
            cur = cur.to_crs(target_crs)
        cur = repair_geometries(cur)
        out[name] = cur
    return out


def discover_point_layers(layers: dict, exclude_names=None):
    exclude_names = set(exclude_names or [])
    candidates = []
    for name, gdf in layers.items():
        if name in exclude_names or len(gdf) == 0:
            continue
        geom_types = set(gdf.geom_type.dropna().unique().tolist())
        if geom_types.issubset({"Point", "MultiPoint"}):
            candidates.append({
                "layer": name,
                "n": int(len(gdf)),
                "geom_types": sorted(list(geom_types)),
            })
    return candidates


def normalize_01(x):
    x = np.asarray(x, dtype=float)
    mn = np.nanmin(x)
    mx = np.nanmax(x)
    if not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
        return np.zeros_like(x, dtype=float)
    return (x - mn) / (mx - mn)


def robust_normalize_01(x, q_low=0.02, q_high=0.98):
    x = np.asarray(x, dtype=float)
    lo = np.nanquantile(x, q_low)
    hi = np.nanquantile(x, q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return normalize_01(x)
    y = (x - lo) / (hi - lo)
    return np.clip(y, 0.0, 1.0)


def distance_to_proximity(distances, scale):
    distances = np.asarray(distances, dtype=float)
    return 1.0 / (1.0 + distances / float(scale))


def smooth_on_regular_grid(values, valid_mask, passes=2):
    arr = np.asarray(values, dtype=float).copy()
    valid = np.asarray(valid_mask, dtype=bool)
    kernel = np.array([[1, 1, 1],
                       [1, 2, 1],
                       [1, 1, 1]], dtype=float)
    arr[~valid] = 0.0
    den_base = valid.astype(float)

    out = arr.copy()
    for _ in range(int(passes)):
        num = convolve2d(out, kernel, mode="same", boundary="fill", fillvalue=0.0)
        den = convolve2d(den_base, kernel, mode="same", boundary="fill", fillvalue=0.0)
        with np.errstate(divide="ignore", invalid="ignore"):
            out = np.divide(
                num,
                den,
                out=np.zeros_like(num, dtype=float),
                where=den > 0
            )
        out[~valid] = np.nan
    return out


def connected_component_filter(binary_arr, min_cells=2):
    arr = np.asarray(binary_arr, dtype=np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, ncomp = label(arr, structure=structure)
    keep = np.zeros_like(arr, dtype=np.uint8)
    for cid in range(1, ncomp + 1):
        comp = (labeled == cid)
        if comp.sum() >= min_cells:
            keep[comp] = 1
    return keep.astype(np.uint8)


def build_grid(mask_gdf: gpd.GeoDataFrame, cell_size=500):
    mask_gdf = repair_geometries(mask_gdf)
    mask_union = unary_union(mask_gdf.geometry)
    minx, miny, maxx, maxy = mask_gdf.total_bounds

    cols = int(math.ceil((maxx - minx) / cell_size))
    rows = int(math.ceil((maxy - miny) / cell_size))

    geoms = []
    rr = []
    cc = []
    for r in range(rows):
        y_top = maxy - r * cell_size
        y_bottom = y_top - cell_size
        for c in range(cols):
            x_left = minx + c * cell_size
            x_right = x_left + cell_size
            cell = box(x_left, y_bottom, x_right, y_top)
            if cell.intersects(mask_union):
                geoms.append(cell)
                rr.append(r)
                cc.append(c)

    grid = gpd.GeoDataFrame(
        {"row": rr, "col": cc},
        geometry=geoms,
        crs=mask_gdf.crs,
    )
    grid["cell_id"] = np.arange(len(grid))
    grid["centroid"] = grid.geometry.centroid
    return grid, (rows, cols), (minx, miny, maxx, maxy)


def compute_distance_feature(grid, source_gdf, scale):
    if len(source_gdf) == 0:
        dist = np.full(len(grid), np.nan, dtype=float)
        prox = np.zeros(len(grid), dtype=float)
        return dist, prox
    union_geom = unary_union(source_gdf.geometry)
    dist = grid["centroid"].distance(union_geom).to_numpy(dtype=float)
    prox = distance_to_proximity(dist, scale)
    return dist, prox

In [ ]:

# =========================
# LOAD DATA
# =========================
layers_raw = load_shapefiles(SHP_DIR)
print("Слои:", sorted(layers_raw.keys()))

mask_name = LAYER_MAP["mask"]
target_crs = choose_target_crs(layers_raw[mask_name])
layers = harmonize_layers(layers_raw, target_crs)

print("Используемые слои:", LAYER_MAP)

point_candidates = discover_point_layers(layers, exclude_names=set(LAYER_MAP.values()))
print("Точечные слои-кандидаты:", point_candidates)

In [ ]:

# =========================
# BUILD GRID
# =========================
grid, GRID_SHAPE, GRID_BOUNDS = build_grid(layers[LAYER_MAP["mask"]], CELL_SIZE)
rows, cols = GRID_SHAPE
minx, miny, maxx, maxy = GRID_BOUNDS

print("Ячеек в сетке:", len(grid))
print("Размер сетки (rows, cols):", GRID_SHAPE)

In [ ]:

# =========================
# FEATURE ENGINEERING
# =========================
for key in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    layer_name = LAYER_MAP[key]
    dist, prox = compute_distance_feature(grid, layers[layer_name], SCALES[key])
    grid[f"dist_{key}"] = dist
    grid[f"prox_{key}"] = prox

grid["tect_combo"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["struct_paleo_combo"] = grid["prox_struct"] * grid["prox_paleo"]
grid["magm_tect_combo"] = grid["prox_magm"] * np.maximum(grid["prox_tect1"], grid["prox_tect2"])
grid["tect_any"] = np.maximum(grid["prox_tect1"], grid["prox_tect2"])

active_flags = []
for nm in ["facies", "paleo", "struct", "magm", "tect1", "tect2"]:
    flag = (grid[f"prox_{nm}"] >= 0.35).astype(int)
    grid[f"flag_{nm}"] = flag
    active_flags.append(f"flag_{nm}")

grid["active_factor_count"] = grid[active_flags].sum(axis=1)

FEATURE_COLS = [
    "prox_facies",
    "prox_paleo",
    "prox_struct",
    "prox_magm",
    "prox_tect1",
    "prox_tect2",
    "tect_combo",
    "struct_paleo_combo",
    "magm_tect_combo",
    "tect_any",
    "active_factor_count",
]

scaler = RobustScaler()
scaled = scaler.fit_transform(grid[FEATURE_COLS].astype(float).fillna(0.0))
scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in FEATURE_COLS], index=grid.index)

for c in scaled_df.columns:
    grid[c] = robust_normalize_01(scaled_df[c].to_numpy())

CHANNEL_COLS = list(scaled_df.columns)
print("Число каналов признаков:", len(CHANNEL_COLS))
CHANNEL_COLS

In [ ]:

# =========================
# MAP TO REGULAR CUBE
# =========================
C = len(CHANNEL_COLS) + 1  # + valid mask channel
cube = np.zeros((rows, cols, C), dtype=np.float32)
valid_mask = np.zeros((rows, cols), dtype=bool)
cell_index_grid = -np.ones((rows, cols), dtype=np.int64)

for idx, rc in grid[["row", "col"]].iterrows():
    r = int(rc["row"])
    c = int(rc["col"])
    cube[r, c, :len(CHANNEL_COLS)] = grid.loc[idx, CHANNEL_COLS].to_numpy(dtype=np.float32)
    cube[r, c, len(CHANNEL_COLS)] = 1.0
    valid_mask[r, c] = True
    cell_index_grid[r, c] = idx

print("Размер матрицы признаков:", cube.shape)

In [ ]:

# =========================
# POSITIVE POINTS -> POSITIVE CELLS
# =========================
exclude = set(LAYER_MAP.values())
candidate_point_names = [x["layer"] for x in point_candidates]

if POSITIVE_LAYER_NAMES is None:
    selected_point_layers = candidate_point_names
else:
    selected_point_layers = [nm for nm in POSITIVE_LAYER_NAMES if nm in candidate_point_names]

positive_points_parts = []
for nm in selected_point_layers:
    part = layers[nm].copy()
    part["source_layer"] = nm
    positive_points_parts.append(part)

if len(positive_points_parts) > 0:
    positive_points = pd.concat(positive_points_parts, ignore_index=True)
    positive_points = gpd.GeoDataFrame(positive_points, geometry="geometry", crs=layers[selected_point_layers[0]].crs)
    positive_points = repair_geometries(positive_points)
else:
    positive_points = gpd.GeoDataFrame({"source_layer": []}, geometry=[], crs=grid.crs)

print("Используем point-слои для positive:", selected_point_layers)
print("Количество positive points:", len(positive_points))

if len(positive_points) > 0:
    join = gpd.sjoin(
        positive_points[["source_layer", "geometry"]],
        grid[["cell_id", "geometry"]],
        how="inner",
        predicate="within"
    )
    positive_cell_ids = np.sort(join["cell_id"].unique())
else:
    positive_cell_ids = np.array([], dtype=int)

grid["is_positive_cell"] = grid["cell_id"].isin(positive_cell_ids).astype(int)
print("Количество positive cells:", int(grid["is_positive_cell"].sum()))

In [ ]:

# =========================
# PATCH EXTRACTION
# =========================
PAD = PATCH_SIZE // 2
cube_padded = np.pad(cube, ((PAD, PAD), (PAD, PAD), (0, 0)), mode="constant", constant_values=0.0)

all_centers = np.argwhere(valid_mask)
print("Patch-центров всего:", len(all_centers))

if len(all_centers) > MAX_TRAIN_PATCHES:
    sel = np.random.choice(len(all_centers), size=MAX_TRAIN_PATCHES, replace=False)
    train_centers = all_centers[sel]
else:
    train_centers = all_centers.copy()

print("Patch-центров для обучения:", len(train_centers))

In [ ]:

# =========================
# SELF-SUPERVISED AUGMENTATIONS
# =========================
def augment_patch(patch):
    x = patch.copy()
    feat = x[:-1]
    mask_ch = x[-1:]

    if np.random.rand() < 0.9:
        noise = np.random.normal(0.0, 0.03, size=feat.shape).astype(np.float32)
        feat = feat + noise

    if np.random.rand() < 0.6:
        scale = np.random.uniform(0.92, 1.08, size=(feat.shape[0], 1, 1)).astype(np.float32)
        feat = feat * scale

    if np.random.rand() < 0.4:
        drop_n = np.random.randint(1, min(3, feat.shape[0]) + 1)
        drop_idx = np.random.choice(feat.shape[0], size=drop_n, replace=False)
        feat[drop_idx] *= np.random.uniform(0.0, 0.4)

    feat = np.clip(feat, 0.0, 1.2)
    x = np.concatenate([feat, mask_ch], axis=0)
    return x.astype(np.float32)


def get_patch_from_center(center_rc):
    r, c = int(center_rc[0]), int(center_rc[1])
    rp = r + PAD
    cp = c + PAD
    patch = cube_padded[rp-PAD:rp+PAD+1, cp-PAD:cp+PAD+1, :]
    patch = np.moveaxis(patch, -1, 0).astype(np.float32)
    return patch

In [ ]:

# =========================
# DATASETS
# =========================
class ContrastivePatchDataset(Dataset):
    def __init__(self, centers):
        self.centers = np.asarray(centers, dtype=int)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        patch = get_patch_from_center(self.centers[idx])
        v1 = augment_patch(patch)
        v2 = augment_patch(patch)
        return torch.from_numpy(v1), torch.from_numpy(v2)


class PatchDataset(Dataset):
    def __init__(self, centers):
        self.centers = np.asarray(centers, dtype=int)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        patch = get_patch_from_center(self.centers[idx])
        return torch.from_numpy(patch), torch.tensor(self.centers[idx], dtype=torch.long)

In [ ]:

# =========================
# MODEL
# =========================
class ContrastiveEncoder(nn.Module):
    def __init__(self, in_channels, embed_dim=32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 96, kernel_size=3, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(96, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.embedding = nn.Linear(128, embed_dim)
        self.projector = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim),
        )

    def encode(self, x):
        h = self.features(x).flatten(1)
        z = self.embedding(h)
        z = F.normalize(z, dim=1)
        return z

    def forward(self, x):
        z = self.encode(x)
        p = self.projector(z)
        p = F.normalize(p, dim=1)
        return z, p


def nt_xent_loss(z1, z2, temperature=0.15):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    N = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)
    sim = torch.mm(z, z.t()) / temperature

    mask = torch.eye(2 * N, dtype=torch.bool, device=sim.device)
    sim = sim.masked_fill(mask, -1e9)

    pos_idx = torch.arange(N, device=sim.device)
    pos = torch.cat([pos_idx + N, pos_idx], dim=0)

    loss = F.cross_entropy(sim, pos)
    return loss

In [ ]:

# =========================
# TRAIN
# =========================
train_ds = ContrastivePatchDataset(train_centers)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

model = ContrastiveEncoder(in_channels=C, embed_dim=EMBED_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
model.train()
for epoch in range(1, EPOCHS + 1):
    losses = []
    for x1, x2 in train_loader:
        x1 = x1.to(DEVICE, non_blocking=True)
        x2 = x2.to(DEVICE, non_blocking=True)

        _, p1 = model(x1)
        _, p2 = model(x2)
        loss = nt_xent_loss(p1, p2, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    mean_loss = float(np.mean(losses)) if len(losses) > 0 else np.nan
    history.append({"epoch": epoch, "train_loss": mean_loss})
    print(f"Epoch {epoch:03d} | contrastive_loss={mean_loss:.6f}")

In [ ]:

# =========================
# EMBEDDINGS FOR ALL VALID CELLS
# =========================
infer_ds = PatchDataset(all_centers)
infer_loader = DataLoader(infer_ds, batch_size=BATCH_SIZE, shuffle=False)

all_embeddings = np.zeros((len(all_centers), EMBED_DIM), dtype=np.float32)
all_center_rows = np.zeros((len(all_centers), 2), dtype=int)

model.eval()
cursor = 0
with torch.no_grad():
    for xb, center_rc in infer_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        z = model.encode(xb).cpu().numpy()
        n = len(z)
        all_embeddings[cursor:cursor+n] = z
        all_center_rows[cursor:cursor+n] = center_rc.numpy()
        cursor += n

embedding_grid = np.full((rows, cols, EMBED_DIM), np.nan, dtype=np.float32)
for emb, (r, c) in zip(all_embeddings, all_center_rows):
    embedding_grid[int(r), int(c), :] = emb

positive_centers = []
for cid in positive_cell_ids:
    rc = grid.loc[grid["cell_id"] == cid, ["row", "col"]].iloc[0]
    positive_centers.append((int(rc["row"]), int(rc["col"])))
positive_centers = np.asarray(positive_centers, dtype=int)

print("Positive centers:", len(positive_centers))

In [ ]:

# =========================
# PROTOTYPE SIMILARITY
# =========================
if len(positive_centers) == 0:
    raise RuntimeError(
        "Не найдено ни одной positive-ячейки. "
        "Проверь POSITIVE_LAYER_NAMES или point-слои."
    )

positive_embeddings = []
for r, c in positive_centers:
    vec = embedding_grid[r, c, :]
    if np.all(np.isfinite(vec)):
        positive_embeddings.append(vec)

positive_embeddings = np.asarray(positive_embeddings, dtype=np.float32)
if len(positive_embeddings) == 0:
    raise RuntimeError("Positive embeddings пусты.")

prototype = positive_embeddings.mean(axis=0)
prototype = prototype / (np.linalg.norm(prototype) + 1e-12)

sim = all_embeddings @ prototype
sim = np.clip(sim, -1.0, 1.0)

prospectivity_valid = robust_normalize_01(sim, 0.01, 0.99)

prospectivity_arr = np.full((rows, cols), np.nan, dtype=np.float32)
for score, (r, c) in zip(prospectivity_valid, all_center_rows):
    prospectivity_arr[int(r), int(c)] = score

prospectivity_arr = smooth_on_regular_grid(prospectivity_arr, valid_mask, passes=SMOOTH_PASSES)
prognoz_arr = 1.0 - prospectivity_arr

In [ ]:

# =========================
# TOP ZONES
# =========================
valid_scores = prospectivity_arr[valid_mask]
q_top = float(np.nanquantile(valid_scores, TOP_ZONE_Q))
local_max = maximum_filter(np.nan_to_num(prospectivity_arr, nan=0.0), size=LOCAL_MAX_SIZE)
peak_mask = (prospectivity_arr >= q_top) & (prospectivity_arr >= local_max - 1e-9) & valid_mask

top_zone_arr = connected_component_filter(peak_mask.astype(np.uint8), min_cells=MIN_TOP_COMPONENT_CELLS)

grid["prospectivity"] = np.nan
grid["prognoz"] = np.nan
grid["top_zone"] = 0
grid["prototype_similarity"] = np.nan

sim_arr = np.full((rows, cols), np.nan, dtype=np.float32)
for s, (r, c) in zip(prospectivity_valid, all_center_rows):
    sim_arr[int(r), int(c)] = s

for idx, rc in grid[["row", "col"]].iterrows():
    r, c = int(rc["row"]), int(rc["col"])
    grid.at[idx, "prospectivity"] = float(prospectivity_arr[r, c])
    grid.at[idx, "prognoz"] = float(prognoz_arr[r, c])
    grid.at[idx, "top_zone"] = int(top_zone_arr[r, c])
    grid.at[idx, "prototype_similarity"] = float(sim_arr[r, c])

In [ ]:

# =========================
# SIMPLE POSITIVE DIAGNOSTICS
# =========================
metrics = {
    "method": "Self-Supervised Contrastive CNN + Prototype Similarity",
    "device": str(DEVICE),
    "grid_rows": int(rows),
    "grid_cols": int(cols),
    "n_cells": int(len(grid)),
    "patch_size": int(PATCH_SIZE),
    "max_train_patches": int(MAX_TRAIN_PATCHES),
    "n_positive_points": int(len(positive_points)),
    "n_positive_cells": int(grid["is_positive_cell"].sum()),
    "embed_dim": int(EMBED_DIM),
    "epochs": int(EPOCHS),
    "selected_point_layers": selected_point_layers,
    "n_top_zone_cells": int(grid["top_zone"].sum()),
}

if grid["is_positive_cell"].sum() > 0:
    pos_scores = grid.loc[grid["is_positive_cell"] == 1, "prospectivity"].to_numpy(dtype=float)
    all_scores = grid["prospectivity"].to_numpy(dtype=float)

    metrics["positive_mean_prospectivity"] = float(np.nanmean(pos_scores))
    metrics["all_mean_prospectivity"] = float(np.nanmean(all_scores))

    for q in [0.90, 0.95, 0.98]:
        thr = float(np.nanquantile(all_scores, q))
        top_mask = grid["prospectivity"] >= thr
        hit = int(grid.loc[top_mask, "is_positive_cell"].sum())
        total_top = int(top_mask.sum())
        metrics[f"hit_count_at_q_{q:.2f}"] = hit
        metrics[f"top_cells_at_q_{q:.2f}"] = total_top

metrics

In [ ]:

# =========================
# SAVE OUTPUTS
# =========================
history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)

grid_to_save = gpd.GeoDataFrame(
    grid.drop(columns=["centroid"], errors="ignore").copy(),
    geometry="geometry",
    crs=grid.crs,
)

if SAVE_FULL_GRID_GPKG:
    grid_to_save.to_file(OUT_DIR / "contrastive_proto_grid.gpkg", driver="GPKG")

top_zone = grid_to_save[grid_to_save["top_zone"] == 1].copy()
if len(top_zone) > 0:
    top_zone.to_file(OUT_DIR / "top_zone_cells.gpkg", driver="GPKG")

if len(positive_points) > 0:
    positive_points.to_file(OUT_DIR / "positive_points.gpkg", driver="GPKG")

grid_to_save.drop(columns="geometry").to_csv(OUT_DIR / "grid_attributes.csv", index=False)

with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

np.save(OUT_DIR / "prototype_vector.npy", prototype)
np.save(OUT_DIR / "embedding_valid.npy", all_embeddings)

print("Сохранено в:", OUT_DIR)

In [ ]:

# =========================
# PLOTS
# =========================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

grid.plot(
    column="prospectivity",
    ax=axes[0],
    cmap="RdYlBu_r",
    linewidth=0,
    legend=True,
    missing_kwds={"color": "lightgray"},
)
layers[LAYER_MAP["mask"]].boundary.plot(ax=axes[0], color="black", linewidth=0.6)
if SHOW_POINTS and len(positive_points) > 0:
    positive_points.plot(ax=axes[0], color="yellow", markersize=12, edgecolor="black", linewidth=0.3)
axes[0].set_title("Prospectivity (Prototype Similarity)")
axes[0].set_axis_off()

grid.plot(
    column="prognoz",
    ax=axes[1],
    cmap="RdYlBu",
    linewidth=0,
    legend=True,
    missing_kwds={"color": "lightgray"},
)
layers[LAYER_MAP["mask"]].boundary.plot(ax=axes[1], color="black", linewidth=0.6)
if len(top_zone) > 0:
    top_zone.boundary.plot(ax=axes[1], color="yellow", linewidth=1.2)
if SHOW_POINTS and len(positive_points) > 0:
    positive_points.plot(ax=axes[1], color="yellow", markersize=12, edgecolor="black", linewidth=0.3)
axes[1].set_title("Prognoz = 1 - prospectivity")
axes[1].set_axis_off()

plt.tight_layout()
plt.savefig(OUT_DIR / "contrastive_proto_result.png", dpi=250, bbox_inches="tight")
plt.show()

In [ ]:

# История обучения
if len(history_df) > 0:
    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_loss"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Contrastive loss")
    plt.title("Training history")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "training_history.png", dpi=200, bbox_inches="tight")
    plt.show()

## Что крутить в первую очередь

Если карта получилась слишком шумной:
- увеличь `SMOOTH_PASSES` до `3`;
- подними `TOP_ZONE_Q` до `0.995`;
- уменьши число каналов до 6 базовых proximity-каналов.

Если сеть учится слишком долго на CPU:
- уменьши `MAX_TRAIN_PATCHES` до `5000`;
- уменьши `EPOCHS` до `15`;
- оставь `PATCH_SIZE = 11`.

Если `positive cells = 0`:
- явно задай `POSITIVE_LAYER_NAMES = ["result", "layer_01"]`
- или проверь, какие point-слои действительно содержат нужные проявления.